# This notebook is to do some initial exploration with how the RAG pipeline is performing

## Imports

In [1]:
import sys
import pickle

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import EnsembleRetriever

# Local imports
path_to_src = "../src/"
sys.path.insert(0, path_to_src)
from rag_pipeline import get_rag_response


/Users/michaeleirikson/miniconda3/envs/find-good-books/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load models

In [3]:
# load searches
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.load_local(
    "../data/processed/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

semantic_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

with open('../data/processed/retriever.pkl', 'rb') as file:
    bm25_retriever = pickle.load(file)

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.2, 0.8]
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13601.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Exploration

### Semantic RAG

In [7]:
user_prompts = [
    "Are there any good books on starting a business?",
    "How to build a log cabin?",
    "Books on mythical beasts like bigfoot and the lock ness monster?",
    "Tell me about books about canadian national parks.",
    "I'm a data science student, can you recommend a good book on statistics?"
]

for user_prompt in user_prompts:
    response = get_rag_response(user_prompt, semantic_retriever)
    print(response)

Yes. '10 Mistakes Startup Businesses Make & How You Can Avoid Them' by Dee Edwards and 'The Practical Guide to a Rapid Start-up (Launch Your Business in 20 Proven Steps)' by Melissa Visconti Moreno cover starting a business.
There are no relevant books available in the sample to answer the question of how to build a log cabin.
A Brave Knight's Quest, Author: , features a knight's encounters with a dragon, the Loch Ness monster, and Bigfoot.
There are no relevant books available in the book sample about Canadian National Parks.
Scammed By Statistics: How we are Lied to, Cheated and Manipulated by Statistics...and why you should care by Edward Zaccaro.


Check the same queries again to see variation in the responses:

In [8]:
for user_prompt in user_prompts:
    response = get_rag_response(user_prompt, semantic_retriever)
    print(response)

Yes, the following books are recommended for starting a business:

1. "The Small Business Owners Handbook" by Martin Sharp
2. "10 Mistakes Startup Businesses Make & How You Can Avoid Them" by Dee Edwards
3. "The Practical Guide to a Rapid Start-up" by Melissa Visconti Moreno
4. "Selling Out to Your Level of Comfort" by Randy King
There are no relevant books available in the sample dataset that provide instructions on how to build a log cabin.
There are no relevant books available in the sample.
There are no relevant books available in the sample.
I highly recommend "Scammed By Statistics: How we are Lied to, Cheated and Manipulated by Statistics...and why you should care" by Edward Zaccaro.


Check for some variations on the prompts, making them simpler for the semantic retriever.

In [9]:
user_prompts2 = [
    "Starting a business",
    "Build log cabin",
    "mythical beasts",
    "canadian national parks",
    "statistics"
]

for user_prompt in user_prompts:
    response = get_rag_response(user_prompt, semantic_retriever)
    print(response)

Yes, there are several good books on starting a business: 
- 10 Mistakes Startup Businesses Make & How You Can Avoid Them by Dee Edwards
- The Practical Guide to a Rapid Start-up (Launch Your Business in 20 Proven Steps) by Melissa Visconti Moreno
- Selling Out to Your Level of Comfort by Randy King
There are no relevant books available in the sample.
A Brave Knight's Quest with no author listed and Unicorns, Myths and Monsters (4) by Anne Marie Ryan mention Bigfoot and Loch Ness monster respectively.
There are no relevant books available in the sample about Canadian national parks.
I recommend "Scammed By Statistics: How we are Lied to, Cheated and Manipulated by Statistics...and why you should care" by Edward Zaccaro. It provides knowledge to properly interpret statistical data and avoids bad decision-making.


### Ensemble RAG

In [10]:
for user_prompt in user_prompts:
    response = get_rag_response(user_prompt, ensemble_retriever)
    print(response)

Yes, here are some relevant books:

1. The Small Business Owners Handbook (Expert Themed Anthology) by Martin Sharp
2. Ready: What to Expect When Starting a Business by Lyndsey Clutteur DePalma
3. The Practical Guide to a Rapid Start-up (Launch Your Business in 20 Proven Steps) by Melissa Visconti Moreno
4. Selling Out to Your Level of Comfort by Randy King
5. 10 Mistakes Startup Businesses Make & How
"There are no relevant books available in the books sample."
A Brave Knight's Quest - This book points out Biblical truths about courage, trust and love as the character moves through an adventure with a dragon, the Loch Ness monster, and Bigfoot.
There are no relevant books available in the sample about Canadian national parks.
Scammed By Statistics: How we are Lied to, Cheated and Manipulated by Statistics...and why you should care, by Edward Zaccaro.


In [11]:
for user_prompt in user_prompts2:
    response = get_rag_response(user_prompt, ensemble_retriever)
    print(response)

"Plan B: The Real Deal Guide to Creating Your Business" by Kathleen Rich-New and "The Practical Guide to a Rapid Start-up" by Melissa Visconti Moreno are relevant books for starting a business.
No Quittin' Sense by C. C. White, however it's not a direct guide on building log cabin. 
But, if you're interested in log cabin construction, you might find "Norwegian Wood: A Tradition of Building" informative.
Unicorns, Myths and Monsters (4) (The Magical Unicorn Society) by Anne Marie Ryan - features various mythical beasts alongside unicorns. 
Legends of Lower Gods: Stories About Creatures From Philippine Mythology & Folklore (Realms of Myths and Reality) - contains stories about Philippine mythical creatures, including demons, dragons, dwarfs, elves, and more.
Man, Myth and Magic: An Illustrated Encyclopedia of the Supernatural (24-Vol. Set) by Cavend
There are no relevant books available in the sample.
Scammed By Statistics: How we are Lied to, Cheated and Manipulated by Statistics...and 

## Findings

The semantic RAG pipeline performs fairly consistently on the two sets of user prompts. Repeating the full sentence prompts gave very similar replies and there wasn't any significant difference between the full sentence prompts and the shortened prompts.

The ensemble RAG pipeline outperforms the purely semantic RAG pipeline here. It seems like there are some relevant books that are not being found using the semantic reliever. 

The LLM seems to be performing quiet well in digesting the augmented prompt and returning sensible replies.